# **Airborne Object Tracking Introspection Module Evaluation**

## **1. Overview**

This notebook evaluates the introspective self-monitoring layer built on top of the HRNet-W32 Tracking-to-Detect (T2D) pipeline, for the [AICrowd Airborne Object Tracking Challenge](https://www.aicrowd.com/challenges/airborne-object-tracking-challenge), a benchmark designed for long-range airborne object detection and tracking in autonomous Sense & Avoid (SAA) systems. Autonomous SAA systems are responsible for maintaining safe separation between aircraft and unforeseen obstacles during flight. While flight paths may be planned in advance, real-world operation in unconstrained airspace introduces the possibility of encountering unexpected airborne objects. In such cases, the system must perform situational awareness, decision-making, and evasive manoeuvre planning in real time. 

The primary objective of this notebook is to evaluate **introspection strategies** layered on top of a fixed HRNet-W32 Tracking-to-Detect (T2D) backbone. Rather than comparing alternative detection backbones, each introspection strategy monitors the output of the existing tracking pipeline and determines, on a per-track and per-frame basis, whether tracking quality is degrading. The underlying detection and tracking pipeline remains unchanged throughout all experiments, ensuring that any differences in performance are attributable solely to the introspection module.

### **1.1. Example Airborne Encounter**

<div align="center">
  <img src="../docs/Images/Data_Viewer_Video.gif" width="700"/>
</div>

<p align="center">
  <em>Example Airborne Encounter from the Evaluation Dataset. The Target Aircraft Occupies Only a Few Pixels During the Early Stages of the Encounter.</em>
</p>


---

<br/>

## **2. Introspection Modules Evaluated**

* **None (Control Condition):** Control baseline that never requests escalation, verifying the introspective predictor behaves identically to the default pipeline.
* **Simple Confidence Threshold:** Escalates a track when detection confidence remains below a static threshold for a fixed window of consecutive frames.
* **Rolling Confidence Trend / EWMA:** Escalates based on confidence trajectory, triggering if either an exponentially smoothed confidence drops below a threshold or its linear trend slope turns sharply negative.
* **Consecutive Coast Count:** Escalates a track once it has been coasted via motion prediction beyond a configured consecutive-frame threshold without a confirmed detection.
* **Kalman Filter NIS:** Statistical consistency check that tracks state dynamics using a constant-velocity Kalman filter and escalates when the Normalised Innovation Squared (NIS) exceeds the 95% Chi-squared threshold.


---

<br/>

## **3. Evaluation Metrics**

### **3.1. Introspection Failure Detection Metrics**

* **AWLT (Average Warning Lead Time):** Time between the start of an unbroken warning run and the confirmed failure onset frame ($\text{lead time} = (\text{onset} - \text{episode start}) / \text{FPS}$). Isolated early warnings followed by silence receive no credit.
* **FNR (False Negative Rate):** The proportion of eligible failed tracks for which the introspection module never raised a warning at any point before or during the failure.
* **FAR (False Alarm Rate):** The proportion of nominal (correct) tracks that received at least one warning during their lifetime (i.e. false alerts raised on reliable tracks).
* **Positional Divergence (Track Drift):** The trend in IoU or centroid distance between a track's reported bounding box and its matched ground-truth target over consecutive frames, capturing sub-threshold drift prior to outright tracking failure.

<br/>

### **3.2. Tracking & Detection Performance Metrics**

* **AFDR (Average Frame Detection Rate):** The average detection rate across all evaluated frames.
* **CEDR (Continuous Encounter Detection Rate):** Replaces windowed EDR by measuring single-track continuity across encounters ($\ge 8$ frames). Computed as the ratio of frames matched by the dominant `track_id` to total target frames in view, heavily penalising fragmented tracking across multiple IDs. Motion-predicted (coasted) frames are excluded to reflect true detection coverage.
* **FTPS (False Tracks Per Sequence):** Replaces raw false-positive rates (FPPI/HFAR) by quantifying entirely spurious tracks ($\ge 8$ frames) exceeding an unmatched-frame threshold. Reported across multiple cutoff strictnesses (**FTPS@50**, **FTPS@80**, and **FTPS@95**) as well as an average (**FTPS Avg**), using 80% as the baseline standard.

<br/>

> ⚠️ ***Note:*** *Inference for all models requires a CUDA-capable GPU. While CPU-based loading of model weights is supported for inspection, full evaluation is computationally intensive.*


---

<br/>

## **4. Dataset**

### **4.1. The AOT-Corruption (AOT-C) Benchmark**

The AOT-Corruption (AOT-C) benchmark modifies the held-out test set of the Airborne Object Tracking (AOT) dataset by uniformly applying synthetically engineered real-world corruptions across fixed severity tiers. It spans weather conditions (fog, rain, low light), sensor artifacts (ISO noise, colour quantisation), and optical defocus blur (near and far focus). 

Aerial object detection models encounter significant domain shifts during operation: high degrees of freedom, high platform velocities, and minute targets occupying only a few pixels, making static decision boundaries vulnerable to environmental degradation. By isolating specific degradation sources (such as sensor noise, which induces more severe precision drops than adverse weather), AOT-C establishes a controlled testbed for evaluating model robustness and guiding safe operational deployment.

<br/>

### **4.2. Temporally Ramped Introspection Dataset**

While AOT-C applies static, uniform degradation across full flight sequences, evaluating introspection requires temporal failure dynamics. Our introspection dataset adopts six corruption mechanisms from AOT-C but introduces continuous temporal ramping:

* **ISO / Sensor Noise:** Poisson-Gaussian noise simulating photon counting limits (shot noise) combined with electronic read noise.
* **Optical Defocus (Near Focus):** Mild Gaussian blur simulating autofocus drift focusing in front of the target depth plane ($\text{Max Kernel} = 45$).
* **Optical Defocus (Far Focus):** Severe Gaussian blur simulating focus beyond the intended focal plane ($\text{Max Kernel} = 70$).
* **Generic Defocus Blur:** Continuous multi-scale defocus modeling gradual optical degradation ($\text{Max Kernel} = 55$).
* **Colour Quantisation:** Bit-depth down-sampling (from 8-bit down to 2-bit), reflecting onboard compression, low bit-depth modes, and transmission bandwidth constraints.
* **Low Light:** Reduced scene illumination (down to $0.55\times$ original luminance) coupled with additive sensor noise, mimicking dusk, dawn, or heavy cloud cover.

Rather than remaining constant, each flight sequence is assigned a randomised onset frame where corruption severity ramps smoothly from $0.0$ to $1.0$ over a defined window before holding at peak. This converts a static robustness benchmark into a structured **early-warning failure testbed**, enabling evaluation of whether an introspection module can warn ahead of a tracking failure rather than simply reacting after track loss occurs.

<br/>

### **4.3. Example Degradation Progression**

<div align="center" style="display: flex; justify-content: center; gap: 20px;">
  <div style="text-align: center;">
    <img src="../docs/Images/Introspection_Progression_Clean.png" width="650"/>
    <p><em>Nominal Pre-Ramp (Severity = 0.0)</em></p>
  </div>
  <div style="text-align: center;">
    <img src="../docs/Images/Introspection_Progression_Ramping.png" width="650"/>
    <p><em>Ramp Midpoint (Severity ≈ 0.5)</em></p>
  </div>
  <div style="text-align: center;">
    <img src="../docs/Images/Introspection_Progression_Peak.png" width="650"/>
    <p><em>Peak Degradation (Severity = 1.0)</em></p>
  </div>
</div>

<p align="center">
  <em>The sequence above illustrates the continuous temporal degradation process across an encounter. Degradation ramps progressively across a defined window, presenting an early-warning signal for introspection modules to detect track decay prior to complete tracking loss.</em>
</p>

---

<br/>

In [ ]:
# Import Required Libraries.
import os
import sys

import cv2
import json
import torch
import random

from glob import glob
import matplotlib.pyplot as plt

# Because Life, Universe & Everything!
random.seed(42)

# Set Working Directory to Project Root.
os.chdir("..") 
PROJECT_ROOT = os.getcwd()

# Add Project Root to Path.
sys.path.insert(0, PROJECT_ROOT)

# Add Core Metric Folders to System Path.
sys.path.insert(0, os.path.join(PROJECT_ROOT, "core"))

# Import Local Project Funtions.
from logging_module import LogLevel, log
from src.data.original_dataset.browse_data import dataLoader
from src.data.original_dataset.download_data import main as downloadDataset
from src.data.corruption_dataset.build_corrupted_dataset import main as corruptDataset
from src.introspection_module.view_introspection_results import generateIntrospectionVideo, spliceVideosSideBySide
from notebooks.helpers.notebook_helpers import (
    get_next_run_dir, load_combined_gt, resolve_local_flights,
    convert_results_to_metrics_format, measure_predictor_complexity, 
    display_styled_summary, build_introspection_summary
)
 
# Confirm Project Root.
log(LogLevel.INFO, "introspection_module:imports", "Project Root Set: %s", PROJECT_ROOT)

In [ ]:
# Choose Which Introspection Module to Run
# --------------------------------------------------
# 0 -> None (Control Condition).
# 1 -> Confidence Threshold (Simple Baseline).
# 2 -> Rolling Confidence Trend / EWMA.
# 3 -> Consecutive-Coast-Count Threshold.
# 4 -> Kalman Filter NIS.
SELECTED_INTROSPECTION_MODULE = 2
module_list = ["none", "confidence_threshold", "rolling_confidence_trend", "coast_count_threshold", "kalman_filter_nis"]

# Select Predictor Architecture Type from Environment (Defaults to HRNet-W32).
PREDICTOR_KIND = os.environ.get("INTROSPECTION_PREDICTOR", "hrnet")   # "hrnet" | "winning"

# Evaluation Filtering.
MIN_SCORE = 0.60
DETECTION_THRESHOLD = 0.60
MIN_TRACK_LEN = 8

# Select the Dataset Split to Evaluate On.
DATASET_SPLIT = os.environ.get("INTROSPECTION_DATASET_SPLIT", "test")

# Set Path to Desired Evaluation Dataset.
DATASET_PATH = f"configs/datasets/training_subsets/{DATASET_SPLIT}.txt"
DATASET_PARTS = ["part1", "part2", "part3"]
DATASET_ROOT = os.path.join(PROJECT_ROOT, "data", "corruption_dataset")

# Determine Base Result Directory Name Based on Selected Predictor Type.
results_name = module_list[SELECTED_INTROSPECTION_MODULE] if PREDICTOR_KIND == "hrnet" else "winning_solution"

# Set Results Variables (Auto-Incrementing Run Directory).
RESULTS_DIR = os.path.join("outputs/results/introspection_module", results_name, DATASET_SPLIT)
run_name = get_next_run_dir(RESULTS_DIR)

# Evaluation Output Folders.
EVAL_GT_DIR = os.path.join(RESULTS_DIR, run_name, "gt")
EVAL_RESULT_DIR  = os.path.join(RESULTS_DIR, run_name, "result")
EVAL_SUMMARY_DIR = os.path.join(RESULTS_DIR, run_name, "summaries")

# Print Summary Statement.
log(LogLevel.INFO, "introspection_module:config", "Configuration Loaded.")
print(f"  Introspection Module : {module_list[SELECTED_INTROSPECTION_MODULE]}")
print(f"  Dataset Split        : {DATASET_SPLIT}")
print(f"  Dataset Root         : {DATASET_ROOT}")
print(f"  Dataset Path         : {DATASET_PATH}")
print(f"  Dataset Parts        : {DATASET_PARTS}")
print(f"  Run Name             : {run_name}")
print(f"  Results Path         : {RESULTS_DIR}")
print(f"  Min Score            : {MIN_SCORE}")
print(f"  Min Track Len        : {MIN_TRACK_LEN}")

In [ ]:
# Download the Required Flight Sequences if Not Already Stored Locally.
downloadDataset(DATASET_PATH)

In [ ]:
# Corrupt the Original Dataset into the Introspection Dataset.
corruptDataset()


---

<br/>

## **5. Data Preparation & Preprocessing**

Before inference can begin, the dataset must be resolved from the flight ID list down to what is actually available on disk, and the ground truth must be filtered to match. This cell performs two steps:

**5.1. Flight Resolution:** Parses the `.txt` flight ID file produced by `data_builder.py`, cross-references each entry against the local `data/<part>/Images/` directories, and builds a lookup of every flight that is both listed and downloaded. Flights in the list that have not been downloaded are logged and skipped.

**5.2. Ground Truth Filtering:** Loads the `groundtruth.json` file from each part's `ImageSets/` folder and merges them into a single structure, retaining only the entries that correspond to locally available flights. Entries for flights not on disk are silently skipped - this is expected, as the ground truth files reference the full AOT dataset rather than just the downloaded subset.

In [ ]:
# Find all Available Sequences from the Provided Dataset.
local_flights = resolve_local_flights(DATASET_PATH, DATASET_ROOT)

# Add Spacing in Console Output for Readability.
print()

# Log How Many Flights were Found & are Ready for Inference.
log(LogLevel.INFO, "introspection_module:preprocessing", "Flights Ready for Inference: %d", len(local_flights))

# Preview the First 5 Flights to Confirm Dataset is Correct.
for name, info in list(local_flights.items())[:5]:
    # Count No. Images in the Flight Directory.
    n_imgs = len(list(info["images_dir"].glob("*.png")))

    # Print Flight Part, Name, & No. Images.
    print(f"  [{info['part']}]  {name}  ({n_imgs} images)")

# If there are More than 5 Flights, Log How Many Additional Ones Exist.
if len(local_flights) > 5:
    log(LogLevel.DEBUG, "introspection_module:preprocessing", " ... and %d more.", len(local_flights) - 5)

# Load & Filter Ground Truth to Include Local Flights Only. 
combined_gt = load_combined_gt(DATASET_PARTS, local_flights, DATASET_ROOT)
log(LogLevel.INFO, "introspection_module:preprocessing", "Flights in Filtered Ground Truth: %d", len(combined_gt['samples']))

# Build Set of Image Filenames That Physically Exist on Disk.
all_images = {name for f in local_flights.values() for name in f["image_names"]}

# Cull GT Entities to Only Those With a Corresponding Image on Disk.
before = sum(len(flight["entities"]) for flight in combined_gt["samples"].values())

for flight_hash, flight_data in combined_gt["samples"].items():
    flight_data["entities"] = [
        e for e in flight_data["entities"]
        if e["img_name"] in all_images
    ]

after = sum(len(flight["entities"]) for flight in combined_gt["samples"].values())

# Report How Many Images were Removed. 
log(LogLevel.INFO, "introspection_module:preprocessing",
    "GT Culled: %d -> %d entities (%d removed for missing images).", before, after, before - after)

# Save the Filtered Ground Truth to the Evaluation Folder.
os.makedirs(EVAL_GT_DIR, exist_ok=True)
gt_out_path = os.path.join(EVAL_GT_DIR, "groundtruth.json")

# Copy the Ground Truth Values to the .json.
with open(gt_out_path, "w") as f:
    json.dump(combined_gt, f)

# Print Success Message.
log(LogLevel.INFO, "introspection_module:preprocessing", "Ground Truth Saved: %s", gt_out_path)


---

<br/>

## **6. Evaluate Models**

### **6.1. Run Inference**

This section runs inference for the selected backbone configuration. The evaluation is controlled via the `SELECTED_MODEL` parameter, which determines which backbone is instantiated and evaluated within the shared detection pipeline. All models are evaluated under a consistent inference setup to ensure fair comparison across backbone architectures, using the same preprocessing, detection pipeline, and evaluation metrics.

<br/>

#### **6.1.1. Prepare Evaluation Directory & Configure Environment Variables.**

In [ ]:
# Create a Directory where all Evaluation Flight Data will be Stored.
EVAL_DATA_DIR = os.path.join(RESULTS_DIR, run_name, "data")
os.makedirs(EVAL_DATA_DIR, exist_ok=True)

# Link Each Local Flight's Image Folder into the Evaluation Directory.
for folder_name, info in local_flights.items():
    link_path = os.path.join(EVAL_DATA_DIR, folder_name)
    if not os.path.exists(link_path):
        os.symlink(info["images_dir"], link_path)

# Set Environment Variables that AirbornePredictor and SelectedModelPredictor Read.
os.environ["TEST_DATASET_PATH"] = EVAL_DATA_DIR
os.environ["INFERENCE_OUTPUT_PATH"] = RESULTS_DIR
os.environ["DATASET_ENV"] = run_name
os.environ["MIN_TRACK_LEN"] = str(MIN_TRACK_LEN)
os.environ["MIN_SCORE"] = str(MIN_SCORE)
os.environ["DETECTION_THRESHOLD"] = str(DETECTION_THRESHOLD)

# Disable the Per-Flight Timeout.
os.environ["INFERENCE_PER_FLIGHT_TIMEOUT_SECONDS"] = "0"

# Limit Inference to Only the Specified Flights.
os.environ["PARTIAL_RUN_FLIGHTS"] = ",".join(local_flights.keys())

# Print Summary Statement.
log(LogLevel.INFO, "introspection_module:inference", "Environment Configured.")
print(f"  Test Data Path     : {EVAL_DATA_DIR}")
print(f"  Flights to Process : {len(local_flights)}")
print(f"  MIN_SCORE          : {MIN_SCORE}")
print(f"  MIN_TRACK_LEN      : {MIN_TRACK_LEN}")


<br/>

#### **6.1.2. Run Model Inference Across All Evaluation Flights Using the Selected Configuration.**

> ⚠️ ***This cells requires a CUDA GPU and will take several hours to complete.*** *Skip this cell if inference has already been run and results are present in `RESULTS_DIRS`.*

In [ ]:
# Import the Introspective Predictor Class & the Introspection Module Registry.
from src.introspection_module.hrnetw32_introspective.hrnetw32_introspective_test import HRNetW32IntrospectivePredictor
from src.introspection_module.hrnetw32_introspective.introspection_modules import INTROSPECTION_MODULE_MAP

# Check GPU is CUDA Enabled.
if torch.cuda.is_available():
    log(LogLevel.INFO, "introspection_module:device_check", "CUDA-Enabled GPU is Available.")
    print(f"  GPU Name: {torch.cuda.get_device_name(0)}.")
    print(f"  CUDA Version: {torch.version.cuda}.")
else:
      log(LogLevel.WARNING, "introspection_module:device_check", "No CUDA-Enabled GPU Found, Using CPU Instead.")

In [ ]:
# Validate Selection.
if SELECTED_INTROSPECTION_MODULE not in INTROSPECTION_MODULE_MAP:
    log(LogLevel.ERROR, "introspection_module:inference", "Invalid SELECTED_INTROSPECTION_MODULE: %s.", SELECTED_INTROSPECTION_MODULE)
    raise ValueError("Please Select a Valid Introspection Module Index.")

# Instantiate the Selected Introspection Module.
introspection_module = INTROSPECTION_MODULE_MAP[SELECTED_INTROSPECTION_MODULE]()

# Initialise the Target Predictor Based on Architecture Selection.
if PREDICTOR_KIND == "winning":
    # Import & Instantiate the Full Ensemble Predictor.
    from src.introspection_module.winning_solution_star.seg_test import WinningSolutionPredictor
    predictor = WinningSolutionPredictor()
else:
    # Instantiate Single-Backbone Predictor & Inject the Introspection Module.
    predictor = HRNetW32IntrospectivePredictor()
    predictor.introspection_module = introspection_module

# Load Models Before Measuring Complexity.
predictor.inference_setup()
    
# Measure the Current Model's Complexity.
print()
complexity = measure_predictor_complexity(predictor)

# Run Warm-Up Compilation Over Dynamic Tensor Shapes.
predictor.warmup()

# Run Inference Over all Flights.
print()
predictor.run()

# Print Summary Statement.
log(LogLevel.INFO, "introspection_module:inference", "Inference Complete.")
print(f"  Raw Results Saved to: {os.path.join(RESULTS_DIR, run_name)}\n")

# Set Results Directories. 
RAW_RESULTS_PATH = os.path.join(RESULTS_DIR, run_name, "result.json")
METRICS_RESULTS_PATH = os.path.join(EVAL_RESULT_DIR, "result.json")

# Ensure Output Directory Exists.
os.makedirs(EVAL_RESULT_DIR, exist_ok=True)

# Copy the Results File.
convert_results_to_metrics_format(RAW_RESULTS_PATH, METRICS_RESULTS_PATH)

# Print Summary Statement.
log(LogLevel.INFO, "introspection_module:inference", "Results Copied to Metrics Directory.")
print(f"  Metrics-Ready Results Stored at : {METRICS_RESULTS_PATH}")


---

<br/>

## **7. Run Metrics Evaluation**

Run the official AICrowd evaluation metrics against the ground truth for each dataset part, saving per-part summaries to `SUMMARIES_DIR`.

In [ ]:
# Locate the Competition Metrics Script.
metrics_script = os.path.join(PROJECT_ROOT, "core/metrics/run_airborne_metrics.py")

# Raw Results Directory Points to the Run Directory so Metrics Can Write Outcomes
# Back to Each result.json.
RAW_RESULTS_DIR = os.path.join(RESULTS_DIR, run_name)

# Call the Competition Metrics Script.
!python3 {metrics_script} \
    --dataset-folder    {EVAL_GT_DIR} \
    --results-folder    {EVAL_RESULT_DIR} \
    --summaries-folder  {EVAL_SUMMARY_DIR} \
    --raw-results-dir   {RAW_RESULTS_DIR} \
    --min-score         {MIN_SCORE} \
    --min-track-len     {MIN_TRACK_LEN}

# Print Message on Completion. 
log(LogLevel.INFO, "introspection_module:evaluation", "Metrics Complete.")


---

<br/>

## **8. Benchmarking Summary**

Display the aggregated statistics across all three parts of the dataset in order to provide a measure of performance on the full hardened dataset. Additionally, a random sample of results is generated using both the ground truth bounding boxes and the predicted ones, enabling a visual analysis of the model’s performance.

In [ ]:
# Find the Summary JSON in the Summaries Directory.
summary_files = glob(os.path.join(EVAL_SUMMARY_DIR, "*.json"))

# Check that the Summary Files Exist.
if not summary_files:
    log(LogLevel.ERROR, "introspection_module:summary", "No Summary Files Found! Run Inference First.")
else:
    # Build Summary DataFrame.
    summary_df = build_introspection_summary(
        summary_json_path = summary_files[0],
        run_dir           = os.path.join(RESULTS_DIR, run_name),
        output_csv_path   = os.path.join(EVAL_SUMMARY_DIR, "introspection_module_benchmark_summary.csv"),
    )

    # Print Results.
    display_styled_summary(summary_df)


---

<br/>

## **9. Introspection Module Evaluation Results**

The table below reports performance across the evaluated introspection strategies on the corrupted dataset. Because the underlying HRNet-W32 (T2D) detection and tracking pipeline is fixed across all experiments, tracking metrics (AFDR, CEDR, FTPS, and Positional Divergence) remain constant, isolating the warning behaviour of each module.

<div align="center">

<table>
  <thead>
    <tr>
      <th style="text-align: center;">Model</th>
      <th style="text-align: center;">AWLT (s)<br><small>(Divergence, Loss)</small></th>
      <th style="text-align: center;">FNR<br><small>(Divergence, Loss)</small></th>
      <th style="text-align: center;">FAR</th>
      <th style="text-align: center;">Positional Divergence<br><small>(Mean IoU Slope, Drift)</small></th>
      <th style="text-align: center;">AFDR</th>
      <th style="text-align: center;">CEDR</th>
      <th style="text-align: center;">FTPS (Avg)</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: center;"><strong>None (Control)</strong></td>
      <td style="text-align: center;">N/A<br><small>(N/A, N/A)</small></td>
      <td style="text-align: center;">1.00<br><small>(1.00, 1.00)</small></td>
      <td style="text-align: center;">0.00</td>
      <td style="text-align: center;">-0.004603, 56.0%</td>
      <td style="text-align: center;">34.89%</td>
      <td style="text-align: center;">24.63%</td>
      <td style="text-align: center;">0.0051</td>
    </tr>
    <tr>
      <td style="text-align: center;"><strong>Static Confidence Threshold</strong><br><small>(Threshold = 0.65, Window = 1)</small></td>
      <td style="text-align: center;">0.40<br><small>(N/A, 0.40)</small></td>
      <td style="text-align: center;">0.1121<br><small>(0.0, 0.1130)</small></td>
      <td style="text-align: center;">0.7442</td>
      <td style="text-align: center;">-0.004603, 56.0%</td>
      <td style="text-align: center;">34.89%</td>
      <td style="text-align: center;">24.63%</td>
      <td style="text-align: center;">0.0051</td>
    </tr>
    <tr>
      <td style="text-align: center;"><strong>Rolling Confidence Trend / EWMA</strong><br><small>(EWMA Thresh = 0.0, Slope Thresh = -0.02)</small></td>
      <td style="text-align: center;">0.20<br><small>(N/A, 0.20)</small></td>
      <td style="text-align: center;">0.0862<br><small>(1.00, 0.0783)</small></td>
      <td style="text-align: center;">0.6512</td>
      <td style="text-align: center;">-0.004603, 56.0%</td>
      <td style="text-align: center;">34.89%</td>
      <td style="text-align: center;">24.63%</td>
      <td style="text-align: center;">0.0051</td>
    </tr>
    <tr>
      <td style="text-align: center;"><strong>Consecutive Coast-Count</strong><br><small>(Threshold = 2)</small></td>
      <td style="text-align: center;">0.10<br><small>(N/A, 0.10)</small></td>
      <td style="text-align: center;">0.4655<br><small>(0.0, 0.4696)</small></td>
      <td style="text-align: center;">0.5116</td>
      <td style="text-align: center;">-0.004603, 56.0%</td>
      <td style="text-align: center;">34.89%</td>
      <td style="text-align: center;">24.63%</td>
      <td style="text-align: center;">0.0051</td>
    </tr>
    <tr>
      <td style="text-align: center;"><strong>Kalman Filter NIS</strong><br><small>(Chi-Squared Threshold = 7.378)</small></td>
      <td style="text-align: center;">0.45<br><small>(N/A, 0.45)</small></td>
      <td style="text-align: center;">0.3276<br><small>(0.5, 0.3304)</small></td>
      <td style="text-align: center;">1.0000</td>
      <td style="text-align: center;">-0.004603, 56.0%</td>
      <td style="text-align: center;">34.89%</td>
      <td style="text-align: center;">24.63%</td>
      <td style="text-align: center;">0.0051</td>
    </tr>
  </tbody>
</table>

</div>

<p align="center">
  <em>AWLT is reported as the median to prevent high-value outlier skew. Evaluated across 116 failed tracks (1 Divergence, 115 Premature Loss) for FNR and 43 correctly tracked sequences for FAR. Positional Divergence is assessed across 166 eligible tracks.</em>
</p>


---

<br/>

## **10. Introspection Module Visualisations**

To complement the aggregate metrics in Sections 8 and 9, this section renders per-flight annotated videos showing exactly how each introspection module tracked a sequence over time, and where it chose to raise a warning. Rather than judging a module purely on summary statistics, this gives a frame-by-frame view of tracking coverage and warning behaviour across a full flight. This is also useful for sanity-checking that a module which looks strong on paper is also behaving sensibly in practice.

In [ ]:
# Set Visualisations Folder Path. 
VISUALISER_DIR = os.path.join(RESULTS_DIR, run_name, "visualisations")

# Ensure Visualisations Directory Exists.
os.makedirs(VISUALISER_DIR, exist_ok=True)

In [ ]:
# Load the AOT-C Dataset Object.
dataset = dataLoader(download=False)

# Load the Corruption Manifest.
CORRUPTION_MANIFEST_PATH = "configs/datasets/corruption_subsets/corruption_manifest.json"

# Open the Manifest File & Read its Contents.
with open(CORRUPTION_MANIFEST_PATH, "r") as f:
    corruption_manifest = json.load(f)

# Create a Lookup for Each Flight.
manifest_lookup = {entry["flight_id"]: entry for entry in corruption_manifest["flights"]}

# Log the Number of Corrupted Flights Loaded.
log(LogLevel.INFO, "introspection_module:visualisation_setup",
    "Loaded Corruption Manifest: %d Corrupted Flight(s) Available for Severity Overlay.",
    len(manifest_lookup))

<br/>

### **10.1. Generate Per-Flight Prediction Videos**

For a selected flight, `generateIntrospectionVideo()` walks every frame of the flight in order *(not just the frames a track happened to be present for)* and overlays that module's predicted detections as bounding boxes, colour-coded by track state *(active detection, coasted, or introspection warning)*. A status banner along the bottom tracks frame progress and escalates colour whenever the module raises a warning, so degraded sequences are immediately visible when scrubbing through the video.

#### **Static Confidence Trend**

In [ ]:
# Static Confidence Trend (Successful Example).
generateIntrospectionVideo(
    flight_id        = "part2acfeb72c54cc4733a31c32fd1c14df7f", 
    result_json_path = os.path.join(RESULTS_DIR, run_name, "part2acfeb72c54cc4733a31c32fd1c14df7f", "result.json"),
    module_name      = "Static Confidence Trend",
    output_path      = os.path.join(VISUALISER_DIR, "part2acfeb72c54cc4733a31c32fd1c14df7f.mp4"),
    dataset          = dataset,
    manifest_entry   = manifest_lookup.get("part2acfeb72c54cc4733a31c32fd1c14df7f")
)

# Static Confidence Trend (Successful Example).
generateIntrospectionVideo(
    flight_id        = "part38640ad5643e94cd8a921cf263aade59a", 
    result_json_path = os.path.join(RESULTS_DIR, run_name, "part38640ad5643e94cd8a921cf263aade59a", "result.json"),
    module_name      = "Static Confidence Trend",
    output_path      = os.path.join(VISUALISER_DIR, "part38640ad5643e94cd8a921cf263aade59a.mp4"),
    dataset          = dataset,
    manifest_entry   = manifest_lookup.get("part38640ad5643e94cd8a921cf263aade59a")
)

# Static Confidence Trend (Failing FNR Example).
generateIntrospectionVideo(
    flight_id        = "part154a62b2bceba487dbaee2c5d959ea25b", 
    result_json_path = os.path.join(RESULTS_DIR, run_name, "part154a62b2bceba487dbaee2c5d959ea25b", "result.json"),
    module_name      = "Static Confidence Trend",
    output_path      = os.path.join(VISUALISER_DIR, "part154a62b2bceba487dbaee2c5d959ea25b.mp4"),
    dataset          = dataset,
    manifest_entry   = manifest_lookup.get("part154a62b2bceba487dbaee2c5d959ea25b")
)

#### **Rolling Confidence Trend**

In [ ]:
# Rolling Confidence Trend (Successful Example).
generateIntrospectionVideo(
    flight_id        = "part2acfeb72c54cc4733a31c32fd1c14df7f", 
    result_json_path = os.path.join(RESULTS_DIR, run_name, "part2acfeb72c54cc4733a31c32fd1c14df7f", "result.json"),
    module_name      = "Rolling Confidence Trend",
    output_path      = os.path.join(VISUALISER_DIR, "part2acfeb72c54cc4733a31c32fd1c14df7f.mp4"),
    dataset          = dataset,
    manifest_entry   = manifest_lookup.get("part2acfeb72c54cc4733a31c32fd1c14df7f")
)

# Rolling Confidence Trend (Successful Example).
generateIntrospectionVideo(
    flight_id        = "part38640ad5643e94cd8a921cf263aade59a", 
    result_json_path = os.path.join(RESULTS_DIR, run_name, "part38640ad5643e94cd8a921cf263aade59a", "result.json"),
    module_name      = "Rolling Confidence Trend",
    output_path      = os.path.join(VISUALISER_DIR, "part38640ad5643e94cd8a921cf263aade59a.mp4"),
    dataset          = dataset,
    manifest_entry   = manifest_lookup.get("part38640ad5643e94cd8a921cf263aade59a")
)

# Rolling Confidence Trend (Failing FNR Example).
generateIntrospectionVideo(
    flight_id        = "part154a62b2bceba487dbaee2c5d959ea25b", 
    result_json_path = os.path.join(RESULTS_DIR, run_name, "part154a62b2bceba487dbaee2c5d959ea25b", "result.json"),
    module_name      = "Rolling Confidence Trend",
    output_path      = os.path.join(VISUALISER_DIR, "part154a62b2bceba487dbaee2c5d959ea25b.mp4"),
    dataset          = dataset,
    manifest_entry   = manifest_lookup.get("part154a62b2bceba487dbaee2c5d959ea25b")
)

#### **Kalman Filter NIS**

In [ ]:
# Kalman Filter NIS (Successful Example).
generateIntrospectionVideo(
    flight_id        = "part2acfeb72c54cc4733a31c32fd1c14df7f", 
    result_json_path = os.path.join(RESULTS_DIR, run_name, "part2acfeb72c54cc4733a31c32fd1c14df7f", "result.json"),
    module_name      = "Kalman Filter NIS",
    output_path      = os.path.join(VISUALISER_DIR, "part2acfeb72c54cc4733a31c32fd1c14df7f.mp4"),
    dataset          = dataset,
    manifest_entry   = manifest_lookup.get("part2acfeb72c54cc4733a31c32fd1c14df7f")
)

# Kalman Filter NIS (Successful Example).
generateIntrospectionVideo(
    flight_id        = "part38640ad5643e94cd8a921cf263aade59a", 
    result_json_path = os.path.join(RESULTS_DIR, run_name, "part38640ad5643e94cd8a921cf263aade59a", "result.json"),
    module_name      = "Kalman Filter NIS",
    output_path      = os.path.join(VISUALISER_DIR, "part38640ad5643e94cd8a921cf263aade59a.mp4"),
    dataset          = dataset,
    manifest_entry   = manifest_lookup.get("part38640ad5643e94cd8a921cf263aade59a")
)

# Kalman Filter NIS (Failing FNR Example).
generateIntrospectionVideo(
    flight_id        = "part154a62b2bceba487dbaee2c5d959ea25b", 
    result_json_path = os.path.join(RESULTS_DIR, run_name, "part154a62b2bceba487dbaee2c5d959ea25b", "result.json"),
    module_name      = "Kalman Filter NIS",
    output_path      = os.path.join(VISUALISER_DIR, "part154a62b2bceba487dbaee2c5d959ea25b.mp4"),
    dataset          = dataset,
    manifest_entry   = manifest_lookup.get("part154a62b2bceba487dbaee2c5d959ea25b")
)

<br/>

### **10.2. Side-by-Side Module Comparison**

`spliceVideosSideBySide()` takes the per-flight videos produced above and composites them into a single comparison video, played back in sync. This makes it straightforward to directly compare how two introspection strategies handle the *same* flight. For example, whether a more conservative module holds the track for longer, or raises a warning earlier than a simpler threshold-based one.

In [ ]:
# Combine Static & Rolling Confidence Flights (Successful Example).
spliceVideosSideBySide(
    video_paths = [
        "results/introspection_module/rolling_confidence_trend/visualisations/successful_warning_example_1.mp4",
        "results/introspection_module/simple_confidence_threshold/visualisations/successful_warning_example_1.mp4",
    ],
    output_path = "docs/Videos/successful_warning_static_vs_rolling_1.mp4",
    labels = ["Rolling Confidence Trend", "Simple Confidence Threshold"],
)

# Combine Static & Rolling Confidence Flights (Successful Example).
spliceVideosSideBySide(
    video_paths = [
        "results/introspection_module/rolling_confidence_trend/visualisations/successful_warning_example_2.mp4",
        "results/introspection_module/simple_confidence_threshold/visualisations/successful_warning_example_2.mp4",
    ],
    output_path = "docs/Videos/successful_warning_static_vs_rolling_2.mp4",
    labels = ["Rolling Confidence Trend", "Simple Confidence Threshold"],
)

# Combine Static & Rolling Confidence Flights (Failing FNR Example).
spliceVideosSideBySide(
    video_paths = [
        "results/introspection_module/rolling_confidence_trend/visualisations/failed_warning_example_1.mp4",
        "results/introspection_module/simple_confidence_threshold/visualisations/failed_warning_example_1.mp4",
    ],
    output_path = "docs/Videos/failed_warning_static_vs_rolling_3.mp4",
    labels = ["Rolling Confidence Trend", "Simple Confidence Threshold"],
)

<br/>

### **10.3. Side-by-Side Comparison Results**

The visual comparison below demonstrates the behavioural difference between the Static Confidence Threshold and Rolling Confidence Trend / EWMA strategies evaluated on the same encounter sequence.

<div align="center">
  <img src="../docs/GIFs/successful_warning_static_vs_rolling_1.gif" width="1000"/>
</div>

<p align="center">
  <em>Synchronised side-by-side comparison of Rolling Confidence Trend (left) versus Simple Confidence Threshold (right) on a successful warning flight sequence. Notice how the trend-based module reacts to sustained degradation and confidence direction rather than isolated threshold dips.</em>
</p>

---